# 1) About

## Purpose
This notebook transforms raw transaction data into a **temporal behavioral dataset** suitable for Machine Learning Modeling. Each row in final dataset repredents one client at one monthly snapshot, with aggregated behavioral features and a **forward lookimg** fraud label.


## Connection to EDA

In the EDA Notebook, we established:

- Dataset contains 2 AML patterns. Fan-in (multiple senders -> one receiver) and cycle (circular flows).
- Only ~0.13% of the transactions are fraudulent -> heavily imbalanced.
- Fraudulent clients demonstrate distinct behavioral signals -> 2x unique senders/ 2x received trx volume, ~60% more transactions.
- Transaction network is sparse (1.3% density), making counterparty count features discriminative.

## Outcome of the NB

- This notebook will transform raw csv files (accounts/transactions/alerts) into a client&month level dataset readt for train-test split.

## Feature Categories

1. Behavioral Intensity: Trx counts, volumes, rolling averages (7d/30d/90)
2. Fan-In SIgnals: Unique senders, sender diversity, received volume
3. Cycle Signals: Sender-receiver overlap, bidirectional behavior patterns
4. Velocity/Change: Activity spikes, dormancy-activity transitions, new counterparties.

# 2) Importd and Data Loading

In [1]:
import pandas as pd, numpy as np 
from pathlib import Path 
import matplotlib.pyplot as plt 
import seaborn as sns

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# import raw data
data_path = Path("../data/raw")

dfs = {}
for f in data_path.glob("*.csv"):
    dfs[f.stem] = pd.read_csv(f)


# handling timestamp - assuming 2020-01-01 as the start date of the timestamp
start_date = pd.to_datetime("2020-01-01")
for i, df in dfs.items():
    if 'TIMESTAMP' in df.columns:
        df['date'] = start_date + pd.to_timedelta(df['TIMESTAMP'], unit='D')


df_accounts = dfs['accounts']
df_transactions = dfs['transactions']
df_alerts = dfs['alerts']   

# Tagging fraudulent accounts
fraudulent_accounts = list(df_accounts[df_accounts['IS_FRAUD'] == 1]['ACCOUNT_ID'].unique())

# General OW
print(f"Accounts: {len(df_accounts):,}")
print(f"Transactions: {len(df_transactions):,}")
print(f"Alerts: {len(df_alerts):,}")
print(f"Fraud accounts: {len(fraudulent_accounts)}")
print(f"Date range: {df_transactions['date'].min().date()} to {df_transactions['date'].max().date()}")


Accounts: 10,000
Transactions: 1,323,234
Alerts: 1,719
Fraud accounts: 1685
Date range: 2020-01-01 to 2020-07-18


# 3) Snapshot Calendar

Define monthly snapshot dates — each one represents a "prediction point" where we ask: 
**will this client exhibit suspicious behavior in the upcoming month?**

**Design rules:**
- Features use **only past data** (up to snapshot date)
- Labels look at the **full calendar month ahead** (snapshot month)
- Need at least **3 full calendar months of history** for rolling features
- Data range: 2020-01-01 to 2020-07-18

This means:
- **First usable snapshot:** April 1 (Jan, Feb, Mar -> 3 full months of history)
- **Last usable snapshot:** June 1 (needs full June for labels; July only has 18 days)

In [2]:
data_start = df_transactions['date'].min()
data_end = df_transactions['date'].max()
print(f"Data covers a period of {(data_end - data_start).days} days")
print(f"Date Range of Transactions: {data_start} to {data_end}")

# Create Monthly Snapshots

# min history - 3 full calendar months
min_history_months = 3

# all month starts
month_starts = pd.date_range(start=data_start, end=data_end, freq = "MS")

# snapshot_dates -> 3 calendar months of history, 1 calendar month of future

snapshot_dates = []
for i in month_starts:
    month_end = i + pd.offsets.MonthEnd(0)
    has_full_month_ahead = month_end <= data_end

    history_start = i - pd.DateOffset(months = min_history_months)
    has_enough_history = history_start >= data_start

    if has_full_month_ahead and has_enough_history:

        snapshot_dates.append(i)

print(f"\nUsable snapshot dates ({len(snapshot_dates)}):")


# print each snapshot date's rolling window history start and prediction date

for i in snapshot_dates:
    history_start = i - pd.DateOffset(months = min_history_months) 
    history_end =  i - pd.Timedelta(days=1)
    prediction_end = i + pd.offsets.MonthEnd(0)
        # print(f"SnapshotDate {i}, uses history from {history_start} to i - , {month_end}")
    print(f" {i.date()}  |  history from: {history_start.date()} to {history_end.date()} |  predicts: {i.date()} until {prediction_end.date()}")


Data covers a period of 199 days
Date Range of Transactions: 2020-01-01 00:00:00 to 2020-07-18 00:00:00

Usable snapshot dates (3):
 2020-04-01  |  history from: 2020-01-01 to 2020-03-31 |  predicts: 2020-04-01 until 2020-04-30
 2020-05-01  |  history from: 2020-02-01 to 2020-04-30 |  predicts: 2020-05-01 until 2020-05-31
 2020-06-01  |  history from: 2020-03-01 to 2020-05-31 |  predicts: 2020-06-01 until 2020-06-30


# 4) Feature Engineering

For each client at each snapshot date, we will compute behavioral futures using transactions that occured before the snapshot date.  



## 4.a Behavioral Intensity

General transaction behavior per client, computed for both **Recent** (previous 1 month) and **Baseline** (2 months before that) windows. Non-overlapping so velocity ratios give clean change signals.

Features per window:
- Transaction counts (sent/received)
- Total volume (sent/received)
- Average transaction size (sent/received)

In [3]:
### Old Code - Calculates Features by Account - first filters the data by account then window

# def compute_intensity_features(df_trx, account_id, window_start, window_end, prefix):
#     """
#     Compute behavioral intensity features for a single account within a time window.
#     prefix: 'recent' or 'baseline' to distinguish the window.
#     """
#     # Sending behavior
#     sent = df_trx[
#         (df_trx["SENDER_ACCOUNT_ID"] == account_id) &
#         (df_trx["date"] >= window_start) &
#         (df_trx["date"] < window_end)
#     ]
    
#     # Receiving behavior
#     received = df_trx[
#         (df_trx["RECEIVER_ACCOUNT_ID"] == account_id) &
#         (df_trx["date"] >= window_start) &
#         (df_trx["date"] < window_end)
#     ]
    
#     return {
#         f"{prefix}_trx_sent": len(sent),
#         f"{prefix}_trx_received": len(received),
#         f"{prefix}_amount_sent": sent["TX_AMOUNT"].sum() if len(sent) > 0 else 0,
#         f"{prefix}_amount_received": received["TX_AMOUNT"].sum() if len(received) > 0 else 0,
#         f"{prefix}_avg_amount_sent": sent["TX_AMOUNT"].mean() if len(sent) > 0 else 0,
#         f"{prefix}_avg_amount_received": received["TX_AMOUNT"].mean() if len(received) > 0 else 0,
#     }

# # Test on one account, one snapshot
# test_account = df_accounts["ACCOUNT_ID"].iloc[5]
# test_snapshot = snapshot_dates[0]
# test_recent_start = test_snapshot - pd.DateOffset(months=1)

# result = compute_intensity_features(df_transactions, test_account, test_recent_start, test_snapshot, "recent")
# print(f"Test account: {test_account}")
# print(f"Snapshot: {test_snapshot.date()}")
# print(f"Window: {test_recent_start.date()} to {test_snapshot.date()}")
# print()
# for k, v in result.items():
#     print(f"  {k}: {v}")


In [4]:
### This function is more optimized, it filters the data by window first, then filters by account
### Old one -  Loop through each account, and each time filter the ENTIRE 1.3M row dataframe:
### New One: Filter 1,300,000 rows by date → 200,000 rows


def compute_intensity_features(df_trx, snapshot, window_start, prefix):
    """
    Compute behavioral intensity features for all accounts within a time window - 
    prefix: last 1 month or 2 months before last month
    Returns a df with one row per account
    """

    # Filter trx per window

    window_trx = df_trx[(df_trx["date"] >= window_start) & (df_trx["date"] < snapshot)]

    # Sending behavior per client

    sent = (
        df_trx.groupby("SENDER_ACCOUNT_ID")
        .agg(
            trx_sent=("TX_ID", "nunique"),
            amt_sent=("TX_AMOUNT", "sum"),
            avg_amt_sent=("TX_AMOUNT", "mean"),
        )
        .rename_axis("ACCOUNT_ID")
    )

    # Receiving behavior per client

    received = (
        df_trx.groupby("RECEIVER_ACCOUNT_ID")
        .agg(
            trx_received=("TX_ID", "nunique"),
            amt_received=("TX_AMOUNT", "sum"),
            avg_amt_received=("TX_AMOUNT", "mean"),
        )
        .rename_axis("ACCOUNT_ID")
    )

    # Combine features

    features = sent.join(received, "ACCOUNT_ID", "outer").fillna(0)
    # #rename cols
    features.columns =[f"{prefix}_{col}" for col in features.columns]
    return features



# Test: compute recent features for April 1 snapshot
test_snapshot = snapshot_dates[0]
test_recent_start = test_snapshot - pd.DateOffset(months=1)

recent_features = compute_intensity_features(df_transactions, test_snapshot, test_recent_start, "recent")
print(f"Snapshot: {test_snapshot.date()}")
print(f"Window: {test_recent_start.date()} to {test_snapshot.date()}")
print(f"Accounts with activity: {len(recent_features)}")
print(f"\nSample (first 5 accounts):")
print(recent_features.head())


Snapshot: 2020-04-01
Window: 2020-03-01 to 2020-04-01
Accounts with activity: 9999

Sample (first 5 accounts):
            recent_trx_sent  recent_amt_sent  recent_avg_amt_sent  \
ACCOUNT_ID                                                          
1                        24          4219.20               175.80   
2                        18          2556.90               142.05   
3                        19          2391.72               125.88   
4                        23          3475.99               151.13   
5                        38          2669.12                70.24   

            recent_trx_received  recent_amt_received  recent_avg_amt_received  
ACCOUNT_ID                                                                     
1                           0.0                 0.00                     0.00  
2                           0.0                 0.00                     0.00  
3                          23.0               313.49                    13.63  
4    

## 4.b Fan-In Signals

Features that capture fan-in signals -> multiple senders funneling money to a single receiver. 

From the EDA, we know that fraudulent clients have 2x as many unique senders and 2x received volume compared to non-fraud clients.

Features:

- Unique Senders: Number of unique accounts sending funds to this account
- Received Volume Concentration: What % of received trx comes from the top sender (A low concentration of MANY senders is suspicious)
- Sender Diversity Ratio: Unique senders / total received trx (lots of different counterparties sending money is suspicious)
- Received-to-send ratio: Amount received/ amount send 

In [5]:
def compute_fanin_features(df_trx, snapshot, window_start, prefix):
    """
    Compute fan-in features for ALL accounts within a time window.
    """
    window_trx = df_trx[(df_trx["date"] >= window_start) & (df_trx["date"] < snapshot)]
    
    # Unique senders per receiver
    unique_senders = window_trx.groupby("RECEIVER_ACCOUNT_ID")["SENDER_ACCOUNT_ID"].nunique()
    
    # Received volume concentration — % of total received amount from top sender
    amt_per_sender = window_trx.groupby(["RECEIVER_ACCOUNT_ID", "SENDER_ACCOUNT_ID"])["TX_AMOUNT"].sum()
    received_concentration = amt_per_sender.groupby(level=0).apply(lambda x: x.max() / x.sum())
    
    # Sender diversity — unique senders / total received transactions
    trx_received = window_trx.groupby("RECEIVER_ACCOUNT_ID")["TX_ID"].count()
    sender_diversity = unique_senders / trx_received
    
    # Received to sent ratio
    total_received = window_trx.groupby("RECEIVER_ACCOUNT_ID")["TX_AMOUNT"].sum()
    total_sent = window_trx.groupby("SENDER_ACCOUNT_ID")["TX_AMOUNT"].sum()
    total_received.index.name = "ACCOUNT_ID"
    total_sent.index.name = "ACCOUNT_ID"
    recv_sent_ratio = total_received / total_sent.reindex(total_received.index, fill_value=1)
    
    # Combine — all naming happens here
    fanin = pd.DataFrame({
        f"{prefix}_unique_senders": unique_senders,
        f"{prefix}_received_concentration": received_concentration,
        f"{prefix}_sender_diversity": sender_diversity,
        f"{prefix}_recv_sent_ratio": recv_sent_ratio,
    })
    fanin.index.name = "ACCOUNT_ID"
    
    return fanin


In [ ]:
# Test on April 1 snapshot, recent window (March)
test_snapshot = snapshot_dates[0]
test_recent_start = test_snapshot - pd.DateOffset(months=1)

fanin_features = compute_fanin_features(df_transactions, test_snapshot, test_recent_start, "recent")
print(f"\nSnapshot: {test_snapshot.date()}")
print(f"\nWindow: {test_recent_start.date()} to {test_snapshot.date()}")
print(f"\nAccounts with fan-in features: {len(fanin_features)}")
print(f"\nSample (first 5):")
print(fanin_features.head())
    



Snapshot: 2020-04-01

Window: 2020-03-01 to 2020-04-01

Accounts with fan-in features: 9778

Sample (first 5):
            recent_unique_senders  recent_received_concentration  \
ACCOUNT_ID                                                         
3                               1                            1.0   
5                               1                            1.0   
6                               1                            1.0   
7                               1                            1.0   
10                              1                            1.0   

            recent_sender_diversity  recent_recv_sent_ratio  
ACCOUNT_ID                                                   
3                          0.333333                0.162417  
5                          0.500000                0.117454  
6                          0.333333                0.803860  
7                          0.333333                0.648389  
10                         0.333333    

## 4.c Cycle Signals

Features that capture circular flow patters (money moving in loops between accounts). From the EDA, we found that in ~95% of cycle alerts, senders become receivers for the same alert id.

Features:

- Sender-receiver-overlap: % of counterparties that are both senders to and receivers from this client
- bidirectional partners: Counts of accounts with both send and receive relationship

In [7]:
df_transactions.columns


Index(['TX_ID', 'SENDER_ACCOUNT_ID', 'RECEIVER_ACCOUNT_ID', 'TX_TYPE',
       'TX_AMOUNT', 'TIMESTAMP', 'IS_FRAUD', 'ALERT_ID', 'date'],
      dtype='str')

In [12]:
def compute_cycle_features(df_trx, snapshot, window_start, prefix):

    """
    compute cycle features for all accounts wihin a given time window
    """

    # trx of interest

    window_trx = df_trx[(df_trx['date'] >= window_start) & (df_trx['date']< snapshot)]

    # For each account find the account_ids the client is sending money to
    sent_to = window_trx.groupby('SENDER_ACCOUNT_ID')['RECEIVER_ACCOUNT_ID'].apply(set) 
    sent_to.index.name = "ACCOUNT_ID"

    # For each account find the account_ids they are receiving money from
    received_from = window_trx.groupby('RECEIVER_ACCOUNT_ID')['SENDER_ACCOUNT_ID'].apply(set)
    received_from.index.name = "ACCOUNT_ID"

    # To ensure sent to and received from have the same index (in case an account didn't receive/send money)
    all_accounts = sent_to.index.union(received_from.index)
    sent_to = sent_to.reindex(all_accounts, fill_value=set())
    received_from = sent_to.reindex(all_accounts, fill_value=set())

    # Find bidirectional partners: the accounts a client is both sending money and receiving money from
    bidirectional = sent_to.combine(received_from, lambda s,r:len(s&r))

    # Overlap ratio: # bidirectional / total counterparties

    total_counterparties = sent_to.combine(received_from, lambda s,r: (len(s|r)))
    overlap_ratio = bidirectional / total_counterparties.replace(0,1) # in case no cps - replace it with 1 to avoid err

    # Combine
    cycle = pd.DataFrame({

        f"{prefix}_bidirectional_partners": bidirectional,
        f"{prefix}_overlap_ratio":overlap_ratio})
    
    cycle.index.name = 'ACCOUNT_ID'

    return cycle


In [21]:
# Test on April 1 Snapshot (for the recent window - full march)

test_snapshot = snapshot_dates[0]
test_recent_start = test_snapshot - pd.DateOffset(months=1)

cycle_features = compute_cycle_features(df_transactions, test_snapshot, test_recent_start, "recent")



print(f"Snapshot: {test_snapshot.date()}")
print(f"Window: {test_recent_start.date()} to {test_snapshot.date()}")
print(f"Accounts with cycle features: {len(cycle_features)}")
print(f"\nSample (first 5):")
print(cycle_features.head())
print(f"\nAccounts with any bidirectional partners: {(cycle_features['recent_bidirectional_partners'] > 0).sum()}")


Snapshot: 2020-04-01
Window: 2020-03-01 to 2020-04-01
Accounts with cycle features: 9995

Sample (first 5):
            recent_bidirectional_partners  recent_overlap_ratio
ACCOUNT_ID                                                     
1                                       1                   1.0
2                                       1                   1.0
3                                       1                   1.0
4                                       1                   1.0
5                                       2                   1.0

Accounts with any bidirectional partners: 9818


In [29]:
cycle_features["is_fraud"] = cycle_features.index.isin(fraudulent_accounts).astype(int)
print(cycle_features.groupby("is_fraud")["recent_bidirectional_partners"].mean())



is_fraud
0    5.019613
1    7.209620
Name: recent_bidirectional_partners, dtype: float64


## 4.d Velocity/ Change

Velocity features compare recent (previous 1 month) to baseline (2 months before that) to detect behavioral shifts.
A sudden change in behavior is often more suspicious than absolute level of activity.

Features:

- `velocity_trx_sent` — recent sent count / baseline avg monthly sent count
- `velocity_trx_received` — recent received count / baseline avg monthly received count
- `velocity_amount_sent` — recent sent amount / baseline avg monthly sent amount
- `velocity_amount_received` — recent received amount / baseline avg monthly received amount
- `velocity_unique_senders` — recent unique senders / baseline avg unique senders
- `new_counterparty_ratio` — % of counterparties in recent window not seen in baseline

In [30]:
baseline_intensity


NameError: name 'baseline_intensity' is not defined

In [ ]:
def compute_velocity_features(recent_intensity, baseline_intensity, recent_fanin, baseline_fanin):

    """
    Compute velocity features by comparing recent and baseline features

    Baseline is divided by 2 to get the avg before the comparison
    """

    idxx = recent_intensity.index

    velocity = pd.DataFrame()

    # 1. Baseline monthly averages 

    bl_avg_trx_sent = baseline_intensity['baseline_trx_sent'] / 2
    bl_avg_trx_received = baseline_intensity['baseline_trx_received'] / 2
    bl_avg_amt_sent = baseline_intensity['baseline_amt_sent'] / 2
    bl_avg_amt_received = baseline_intensity['baseline_amt_received'] / 2
    bl_avg_unique_senders = baseline_fanin['baseline_unique_senders']/2

    # 2. Compute Velocity Ratios ---- replace 0s with 1 to avoid division err

    velocity["velocity_trx_sent"] = recent_intensity["recent_trx_sent"] / bl_avg_trx_sent.reindex(idx, fill_value=1).replace(0, 1)
    velocity["velocity_trx_received"] = recent_intensity["recent_trx_received"] / bl_avg_trx_received.reindex(idx, fill_value=1).replace(0, 1)
    velocity["velocity_amt_sent"] = recent_intensity["recent_amt_sent"] / bl_avg_amt_sent.reindex(idx, fill_value=1).replace(0, 1)
    velocity["velocity_amt_received"] = recent_intensity["recent_amt_received"] / bl_avg_amt_received.reindex(idx, fill_value=1).replace(0, 1)
    velocity["velocity_unique_senders"] = recent_fanin["recent_unique_senders"] / bl_avg_unique_senders.reindex(idx, fill_value=1).replace(0, 1)


    velocity.index.name = "ACCOUNT_ID"

    return velocity




